## Option 4: Generate Stop-List

## Step 1: Install and import libraries

I'm using `wikipedia-api` to download pages, `re` for text cleaning, and `Counter` from the standard library to count word frequencies.

In [1]:
!pip install wikipedia-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 1.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
import wikipediaapi
import os
import re
from collections import Counter
import matplotlib.pyplot as plt

print("Libraries loaded successfully.")

Libraries loaded successfully.


## Step 2: Download Wikipedia pages

I chose 55 pages across different topics (science, history, geography, biology, arts, economics) to make sure the corpus is diverse.

In [3]:
wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='IR_Course_Assignment/1.0'
)

page_titles = [
    # Science & Technology
    "Artificial intelligence", "Machine learning", "Deep learning", "Computer science",
    "Quantum computing", "Internet", "Algorithm", "Data structure", "Python (programming language)",
    "Robotics",
    # History
    "World War II", "Ancient Rome", "French Revolution", "Renaissance", "Byzantine Empire",
    "Ottoman Empire", "Cold War", "American Civil War", "Industrial Revolution", "Ancient Egypt",
    # Geography
    "Amazon River", "Mount Everest", "Sahara", "Atlantic Ocean", "Europe",
    "Africa", "Australia", "Himalayas", "Mediterranean Sea", "Amazon rainforest",
    # Biology & Medicine
    "DNA", "Evolution", "Human brain", "Cell (biology)", "Photosynthesis",
    "Cancer", "Immune system", "Bacteria", "Virus", "Ecology",
    # Arts & Culture
    "William Shakespeare", "Ludwig van Beethoven", "Leonardo da Vinci", "Philosophy",
    "Literature", "Cinema", "Music", "Architecture", "Painting", "Mythology",
    # Economy & Society
    "Economics", "Democracy", "Globalization", "Climate change", "United Nations"
]

print(f"Planning to download {len(page_titles)} pages")

Planning to download 55 pages


In [4]:
# Save each page as a .txt file in a local folder
os.makedirs("wikipedia_docs", exist_ok=True)

downloaded = []
failed = []

for title in page_titles:
    page = wiki.page(title)
    if page.exists():
        filename = f"wikipedia_docs/{title.replace(' ', '_').replace('/', '_')}.txt"
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(page.text)
        downloaded.append(title)
        print(f"Downloaded: {title}")
    else:
        failed.append(title)
        print(f"Not found: {title}")

print(f"\nTotal downloaded: {len(downloaded)}")
if failed:
    print(f"Failed pages: {failed}")

Downloaded: Artificial intelligence
Downloaded: Machine learning
Downloaded: Deep learning
Downloaded: Computer science
Downloaded: Quantum computing
Downloaded: Internet
Downloaded: Algorithm
Downloaded: Data structure
Downloaded: Python (programming language)
Downloaded: Robotics
Downloaded: World War II
Downloaded: Ancient Rome
Downloaded: French Revolution
Downloaded: Renaissance
Downloaded: Byzantine Empire
Downloaded: Ottoman Empire
Downloaded: Cold War
Downloaded: American Civil War
Downloaded: Industrial Revolution
Downloaded: Ancient Egypt
Downloaded: Amazon River
Downloaded: Mount Everest
Downloaded: Sahara
Downloaded: Atlantic Ocean
Downloaded: Europe
Downloaded: Africa
Downloaded: Australia
Downloaded: Himalayas
Downloaded: Mediterranean Sea
Downloaded: Amazon rainforest
Downloaded: DNA
Downloaded: Evolution
Downloaded: Human brain
Downloaded: Cell (biology)
Downloaded: Photosynthesis
Downloaded: Cancer
Downloaded: Immune system
Downloaded: Bacteria
Downloaded: Virus
Downlo

## Step 3: Tokenize the text

For each document I:
1. Convert everything to **lowercase** (so "The" and "the" count as the same word)
2. Extract only alphabetic words using a regular expression (removes numbers, punctuation, etc.)

Then I combine all words from all documents into a single list.

In [5]:
def tokenize(text):
    text = text.lower()
    words = re.findall(r'[a-z]+', text)
    return words


all_words = []
doc_count = 0

for filename in os.listdir("wikipedia_docs"):
    if filename.endswith(".txt"):
        filepath = os.path.join("wikipedia_docs", filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
        words = tokenize(text)
        all_words.extend(words)
        doc_count += 1

print(f"Documents processed: {doc_count}")
print(f"Total word tokens:   {len(all_words):,}")
print(f"Unique word types:   {len(set(all_words)):,}")
print(f"\nFirst 20 tokens as a sanity check: {all_words[:20]}")

Documents processed: 55
Total word tokens:   502,170
Unique word types:   30,739

First 20 tokens as a sanity check: ['africa', 'is', 'the', 'world', 's', 'second', 'largest', 'and', 'second', 'most', 'populous', 'continent', 'after', 'asia', 'at', 'about', 'million', 'km', 'million', 'square']


## Step 4: Count word frequencies

Using `Counter` I count how many times each word appears across the entire corpus.

In [6]:
word_freq = Counter(all_words)

print(f"Total unique words in corpus: {len(word_freq):,}")
print(f"\nQuick preview – top 10 most frequent words:")
for word, count in word_freq.most_common(10):
    print(f"  {word}: {count:,}")

Total unique words in corpus: 30,739

Quick preview – top 10 most frequent words:
  the: 37,513
  of: 20,157
  and: 17,624
  in: 13,373
  to: 10,941
  a: 9,264
  as: 4,948
  is: 4,589
  by: 4,253
  that: 3,973


## Step 5: Save the stop-list to a file

I take the top 50 words and write them to `stop_list.txt`. The format is: one word per line with its frequency count.

In [7]:
top_50 = word_freq.most_common(50)
stop_words = [word for word, count in top_50]

with open("stop_list.txt", 'w', encoding='utf-8') as f:
    f.write("# Stop List – generated from 55 Wikipedia pages\n")
    f.write(f"# Corpus: {len(all_words):,} tokens across {doc_count} documents\n\n")
    for word, count in top_50:
        f.write(f"{word}\t{count}\n")

print("Saved stop_list.txt")
print(f"Stop-list contains {len(stop_words)} words.")

Saved stop_list.txt
Stop-list contains 50 words.


## Step 6: Display the 50 most common words

Here's the full stop-list with rank, word, frequency, and percentage of the total corpus.

In [8]:
total_tokens = len(all_words)

print("=" * 58)
print(f"{'Rank':<6} {'Word':<20} {'Frequency':>10} {'% of corpus':>12}")
print("=" * 58)

for rank, (word, count) in enumerate(top_50, start=1):
    pct = (count / total_tokens) * 100
    print(f"{rank:<6} {word:<20} {count:>10,} {pct:>11.3f}%")

print("=" * 58)

top50_total = sum(count for _, count in top_50)
coverage = (top50_total / total_tokens) * 100
print(f"\nThese 50 words cover {coverage:.1f}% of all tokens in the corpus.")

Rank   Word                  Frequency  % of corpus
1      the                      37,513       7.470%
2      of                       20,157       4.014%
3      and                      17,624       3.510%
4      in                       13,373       2.663%
5      to                       10,941       2.179%
6      a                         9,264       1.845%
7      as                        4,948       0.985%
8      is                        4,589       0.914%
9      by                        4,253       0.847%
10     that                      3,973       0.791%
11     for                       3,493       0.696%
12     was                       3,214       0.640%
13     with                      3,104       0.618%
14     s                         2,960       0.589%
15     on                        2,880       0.574%
16     are                       2,809       0.559%
17     from                      2,728       0.543%
18     or                        2,314       0.461%
19     which